# Neo4j Knowledge Unit Editor

Edit knowledge units in Neo4j with bi-directional sync to Obsidian vault.

**Current Schema (October 2025):**
- version, type, uid, title
- content (single markdown field)
- domain, quality_score, complexity
- tags, prerequisites, enables, related_to

In [ ]:
# Setup and imports
import asyncio
import json
import os
from pathlib import Path
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
from neo4j import AsyncGraphDatabase

# Import our sync service
import sys
sys.path.append('/home/mike/skuel0')
from core.services.jupyter_neo4j_sync import JupyterNeo4jSync, ConflictResolution

# Configuration — reads from environment, with development defaults
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
VAULT_PATH = Path("/home/mike/0bsidian/skuel")

if not NEO4J_PASSWORD:
    try:
        from core.config.credential_store import get_credential
        NEO4J_PASSWORD = get_credential("NEO4J_PASSWORD", fallback_to_env=True)
    except Exception:
        raise RuntimeError("NEO4J_PASSWORD not set. Export it or use the credential store.")

# Initialize driver and sync service
driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
sync_service = JupyterNeo4jSync(
    driver=driver,
    vault_path=VAULT_PATH,
    conflict_strategy=ConflictResolution.MANUAL
)

print("✅ Setup complete")

## 1. Search and Select Knowledge Unit

In [ ]:
async def search_knowledge_units(query: str = ""):
    """Search for knowledge units in Neo4j."""
    search_query = """
    MATCH (ku:KnowledgeUnit)
    WHERE ku.title CONTAINS $query OR ku.uid CONTAINS $query
    RETURN ku.uid as uid, ku.title as title, ku.domain as domain, ku.complexity as complexity
    ORDER BY ku.title
    LIMIT 20
    """
    
    async with driver.session() as session:
        result = await session.run(search_query, {"query": query})
        records = await result.data()
        return records

# Search widget
search_input = widgets.Text(
    placeholder='Search by title or UID...',
    description='Search:',
    style={'description_width': 'initial'}
)

search_output = widgets.Output()
selected_ku = widgets.Dropdown(
    options=[],
    description='Select:',
    style={'description_width': 'initial'}
)

def on_search_change(change):
    with search_output:
        search_output.clear_output()
        results = asyncio.run(search_knowledge_units(change['new']))
        
        options = [(f"{r['title']} ({r['uid']}) - {r.get('complexity', 'unknown')}", r['uid']) for r in results]
        selected_ku.options = options
        
        if results:
            print(f"Found {len(results)} knowledge units")

search_input.observe(on_search_change, names='value')

display(search_input)
display(selected_ku)
display(search_output)

## 2. Load and Edit Content

Current structure: version, type, uid, title, content, domain, quality_score, complexity, tags, prerequisites, enables, related_to

In [ ]:
# Content editor widgets
title_input = widgets.Text(
    description='Title:', 
    style={'description_width': 'initial'}, 
    layout={'width': '100%'}
)

domain_input = widgets.Text(
    description='Domain:', 
    placeholder='personal, technology, etc.',
    style={'description_width': 'initial'}
)

complexity_input = widgets.Dropdown(
    options=['basic', 'intermediate', 'advanced'],
    description='Complexity:',
    style={'description_width': 'initial'}
)

quality_input = widgets.FloatSlider(
    value=0.85,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Quality:',
    style={'description_width': 'initial'}
)

content_editor = widgets.Textarea(
    description='Content (Markdown):', 
    layout={'width': '100%', 'height': '400px'},
    style={'description_width': 'initial'}
)

tags_input = widgets.Text(
    description='Tags (comma-separated):', 
    style={'description_width': 'initial'}, 
    layout={'width': '100%'}
)

prerequisites_input = widgets.Text(
    description='Prerequisites (UIDs, comma-separated):', 
    style={'description_width': 'initial'}, 
    layout={'width': '100%'}
)

enables_input = widgets.Text(
    description='Enables (UIDs, comma-separated):', 
    style={'description_width': 'initial'}, 
    layout={'width': '100%'}
)

related_input = widgets.Text(
    description='Related (UIDs, comma-separated):', 
    style={'description_width': 'initial'}, 
    layout={'width': '100%'}
)

# Metadata display
metadata_output = widgets.Output()

# Current data holder
current_data = {}

async def load_content():
    """Load selected knowledge unit for editing."""
    global current_data
    
    if not selected_ku.value:
        print("Please select a knowledge unit first")
        return
    
    # Load from Neo4j
    query = """
    MATCH (ku:KnowledgeUnit {uid: $uid})
    OPTIONAL MATCH (ku)-[:PREREQUISITE]->(prereq)
    OPTIONAL MATCH (ku)-[:ENABLES]->(enabled)
    OPTIONAL MATCH (ku)-[:RELATED_TO]->(related)
    RETURN ku,
           collect(DISTINCT prereq.uid) as prerequisites,
           collect(DISTINCT enabled.uid) as enables,
           collect(DISTINCT related.uid) as related_to
    """
    
    async with driver.session() as session:
        result = await session.run(query, {"uid": selected_ku.value})
        record = await result.single()
        
        if not record:
            print(f"Knowledge unit not found: {selected_ku.value}")
            return
        
        ku_data = dict(record["ku"])
        current_data = ku_data
        
        # Populate editors
        title_input.value = ku_data.get('title', '')
        domain_input.value = ku_data.get('domain', 'personal')
        complexity_input.value = ku_data.get('complexity', 'basic')
        quality_input.value = ku_data.get('quality_score', 0.85)
        content_editor.value = ku_data.get('content', '')
        
        # Relationships
        tags_input.value = ', '.join(ku_data.get('tags', []))
        prerequisites_input.value = ', '.join([uid for uid in record['prerequisites'] if uid])
        enables_input.value = ', '.join([uid for uid in record['enables'] if uid])
        related_input.value = ', '.join([uid for uid in record['related_to'] if uid])
        
        # Display metadata
        with metadata_output:
            metadata_output.clear_output()
            print("\n📊 Metadata:")
            print(f"UID: {ku_data.get('uid')}")
            print(f"Version: {ku_data.get('version', '1.0')}")
            print(f"Type: {ku_data.get('type', 'KnowledgeUnit')}")
            print(f"Quality Score: {ku_data.get('quality_score', 'N/A')}")
            print(f"Complexity: {ku_data.get('complexity', 'N/A')}")

# Load button
load_button = widgets.Button(description='Load Selected', button_style='primary')
load_button.on_click(lambda b: asyncio.run(load_content()))

display(load_button)
display(widgets.VBox([
    title_input,
    widgets.HBox([domain_input, complexity_input, quality_input]),
    content_editor,
    tags_input,
    prerequisites_input,
    enables_input,
    related_input,
    metadata_output
]))

## 3. Save Changes to Neo4j

In [ ]:
save_output = widgets.Output()

async def save_changes():
    """Save edited content back to Neo4j."""
    if not current_data:
        print("No content loaded")
        return
    
    uid = current_data.get('uid')
    
    # Update query matching current schema
    update_query = """
    MATCH (ku:KnowledgeUnit {uid: $uid})
    SET ku.title = $title,
        ku.content = $content,
        ku.domain = $domain,
        ku.complexity = $complexity,
        ku.quality_score = $quality_score,
        ku.tags = $tags,
        ku.last_modified = datetime(),
        ku.last_editor = 'jupyter'
    RETURN ku
    """
    
    # Parse lists
    tags = [t.strip() for t in tags_input.value.split(',') if t.strip()]
    prereqs = [p.strip() for p in prerequisites_input.value.split(',') if p.strip()]
    enables = [e.strip() for e in enables_input.value.split(',') if e.strip()]
    related = [r.strip() for r in related_input.value.split(',') if r.strip()]
    
    try:
        async with driver.session() as session:
            # Update node properties
            result = await session.run(update_query, {
                "uid": uid,
                "title": title_input.value,
                "content": content_editor.value,
                "domain": domain_input.value,
                "complexity": complexity_input.value,
                "quality_score": quality_input.value,
                "tags": tags
            })
            
            # Update relationships
            # Clear existing relationships
            await session.run("""
                MATCH (ku:KnowledgeUnit {uid: $uid})-[r:PREREQUISITE|ENABLES|RELATED_TO]->()
                DELETE r
            """, {"uid": uid})
            
            # Create new relationships
            for prereq_uid in prereqs:
                await session.run("""
                    MATCH (ku:KnowledgeUnit {uid: $uid})
                    MATCH (prereq:KnowledgeUnit {uid: $prereq_uid})
                    CREATE (ku)-[:PREREQUISITE]->(prereq)
                """, {"uid": uid, "prereq_uid": prereq_uid})
            
            for enable_uid in enables:
                await session.run("""
                    MATCH (ku:KnowledgeUnit {uid: $uid})
                    MATCH (enabled:KnowledgeUnit {uid: $enable_uid})
                    CREATE (ku)-[:ENABLES]->(enabled)
                """, {"uid": uid, "enable_uid": enable_uid})
            
            for related_uid in related:
                await session.run("""
                    MATCH (ku:KnowledgeUnit {uid: $uid})
                    MATCH (related:KnowledgeUnit {uid: $related_uid})
                    CREATE (ku)-[:RELATED_TO]->(related)
                """, {"uid": uid, "related_uid": related_uid})
            
            with save_output:
                save_output.clear_output()
                print("✅ Saved successfully!")
                print(f"Updated: {uid}")
                print(f"Relationships: {len(prereqs)} prerequisites, {len(enables)} enables, {len(related)} related")
    
    except Exception as e:
        with save_output:
            save_output.clear_output()
            print(f"❌ Save failed: {e}")

save_button = widgets.Button(description='Save to Neo4j', button_style='success')
save_button.on_click(lambda b: asyncio.run(save_changes()))

display(save_button)
display(save_output)

## 4. Sync to Obsidian (Optional)

Export changes back to YAML files in Obsidian vault.

In [ ]:
sync_output = widgets.Output()

async def sync_to_yaml():
    """Export current knowledge unit to YAML file."""
    if not current_data:
        print("No content loaded")
        return
    
    import yaml
    
    uid = current_data.get('uid')
    
    # Load current state from Neo4j
    query = """
    MATCH (ku:KnowledgeUnit {uid: $uid})
    OPTIONAL MATCH (ku)-[:PREREQUISITE]->(prereq)
    OPTIONAL MATCH (ku)-[:ENABLES]->(enabled)
    OPTIONAL MATCH (ku)-[:RELATED_TO]->(related)
    RETURN ku,
           collect(DISTINCT prereq.uid) as prerequisites,
           collect(DISTINCT enabled.uid) as enables,
           collect(DISTINCT related.uid) as related_to
    """
    
    async with driver.session() as session:
        result = await session.run(query, {"uid": uid})
        record = await result.single()
        
        if not record:
            print(f"Knowledge unit not found: {uid}")
            return
        
        ku_data = dict(record["ku"])
        
        # Build YAML structure matching current schema
        yaml_data = {
            'version': ku_data.get('version', '1.0'),
            'type': ku_data.get('type', 'KnowledgeUnit'),
            'uid': ku_data.get('uid'),
            'title': ku_data.get('title'),
            'content': ku_data.get('content', ''),
            'domain': ku_data.get('domain', 'personal'),
            'quality_score': ku_data.get('quality_score', 0.85),
            'complexity': ku_data.get('complexity', 'basic'),
            'tags': ku_data.get('tags', []),
            'prerequisites': [p for p in record['prerequisites'] if p],
            'enables': [e for e in record['enables'] if e],
            'related_to': [r for r in record['related_to'] if r]
        }
        
        # Determine output path
        filename = uid.replace(':', '_') + '.yaml'
        output_path = VAULT_PATH / 'domains' / ku_data.get('domain', 'personal') / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Write YAML
        with open(output_path, 'w') as f:
            yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
        
        with sync_output:
            sync_output.clear_output()
            print(f"✅ Exported to: {output_path}")
            print(f"File size: {output_path.stat().st_size} bytes")

sync_button = widgets.Button(description='Export to YAML', button_style='info')
sync_button.on_click(lambda b: asyncio.run(sync_to_yaml()))

display(sync_button)
display(sync_output)

## 5. Advanced: Cypher Query Editor

In [ ]:
# Direct Cypher query execution for advanced editing
cypher_editor = widgets.Textarea(
    value="MATCH (ku:KnowledgeUnit) RETURN ku.title, ku.uid, ku.complexity LIMIT 5",
    description='Cypher Query:',
    layout={'width': '100%', 'height': '150px'},
    style={'description_width': 'initial'}
)

query_output = widgets.Output()

async def execute_cypher():
    """Execute custom Cypher query."""
    query = cypher_editor.value
    
    try:
        async with driver.session() as session:
            result = await session.run(query)
            records = await result.data()
            
            with query_output:
                query_output.clear_output()
                if records:
                    import pandas as pd
                    df = pd.DataFrame(records)
                    display(df)
                else:
                    print("Query returned no results")
    except Exception as e:
        with query_output:
            query_output.clear_output()
            print(f"Query error: {e}")

execute_button = widgets.Button(description='Execute Query', button_style='primary')
execute_button.on_click(lambda b: asyncio.run(execute_cypher()))

display(cypher_editor)
display(execute_button)
display(query_output)

## 6. Cleanup

In [ ]:
# Close driver when done
await driver.close()
print("Driver closed")